In [2]:
import os, zipfile
import numpy as np
import tensorflow as tf
import cv2
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

from google.colab import drive
drive.mount("/content/drive")

zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"
print("Extracted:", os.listdir(BASE))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Extracted: ['Hand Pd', 'New hand pd Dataset']


In [3]:
!pip install imagehash tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.1 MB/s eta 0:00:00


In [4]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tqdm import tqdm
from PIL import Image
import imagehash
from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split

In [5]:
NEWHANDPD_ROOT = os.path.join(BASE, "New hand pd Dataset")

HEALTHY_ROOT = os.path.join(NEWHANDPD_ROOT, "Healthy")
PARK_ROOT = os.path.join(NEWHANDPD_ROOT, "Parkinson's")

image_paths = []
labels = []

def collect_images(root, label):
    for r, d, files in os.walk(root):
        for f in files:
            if f.lower().endswith((".png",".jpg",".jpeg",".bmp")):
                image_paths.append(os.path.join(r,f))
                labels.append(label)

collect_images(HEALTHY_ROOT,0)
collect_images(PARK_ROOT,1)

print("Raw total:",len(image_paths))
print("Distribution:",Counter(labels))

Raw total: 512
Distribution: Counter({1: 279, 0: 233})


In [6]:
def get_phash(img_path):
    try:
        img = Image.open(img_path).convert("L").resize((224,224))
        return imagehash.phash(img)
    except:
        return None

PHASH_DISTANCE_THRESHOLD = 4

hash_to_paths = defaultdict(list)

for p in tqdm(image_paths):
    h = get_phash(p)
    if h is not None:
        hash_to_paths[str(h)].append(p)

hash_list = list(hash_to_paths.keys())
used=set()
final_keep=[]

for i in tqdm(range(len(hash_list))):

    if hash_list[i] in used:
        continue

    base_hash=imagehash.hex_to_hash(hash_list[i])
    used.add(hash_list[i])
    final_keep.append(hash_list[i])

    for j in range(i+1,len(hash_list)):

        if hash_list[j] in used:
            continue

        compare=imagehash.hex_to_hash(hash_list[j])

        if base_hash-compare<=PHASH_DISTANCE_THRESHOLD:
            used.add(hash_list[j])

path_to_label=dict(zip(image_paths,labels))

clean_paths=[]
clean_labels=[]

for h in final_keep:
    p=hash_to_paths[h][0]
    clean_paths.append(p)
    clean_labels.append(path_to_label[p])

print("Clean images:",len(clean_paths))
print("Distribution:",Counter(clean_labels))

100%|██████████| 467/467 [00:02<00:00, 179.02it/s]

Clean images: 368
Distribution: Counter({0: 190, 1: 178})


In [7]:
paths=np.array(clean_paths)
labels=np.array(clean_labels)

train_paths,temp_paths,y_train,y_temp=train_test_split(
    paths,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=42
)

val_paths,test_paths,y_val,y_test=train_test_split(
    temp_paths,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("Train:",Counter(y_train))
print("Val:",Counter(y_val))
print("Test:",Counter(y_test))

Train: Counter({np.int64(0): 152, np.int64(1): 142})
Val: Counter({np.int64(0): 19, np.int64(1): 18})
Test: Counter({np.int64(0): 19, np.int64(1): 18})


In [8]:
import random

TARGET=2000

expanded_paths=list(train_paths)
expanded_labels=list(y_train)

while len(expanded_paths)<TARGET:

    idx=random.randint(0,len(train_paths)-1)

    expanded_paths.append(train_paths[idx])
    expanded_labels.append(y_train[idx])

print("Expanded train size:",len(expanded_paths))
print("Distribution:",Counter(expanded_labels))

Expanded train size: 2000
Distribution: Counter({np.int64(0): 1057, np.int64(1): 943})


In [9]:
import random

TARGET=2000

expanded_paths=list(train_paths)
expanded_labels=list(y_train)

while len(expanded_paths)<TARGET:

    idx=random.randint(0,len(train_paths)-1)

    expanded_paths.append(train_paths[idx])
    expanded_labels.append(y_train[idx])

print("Expanded train size:",len(expanded_paths))
print("Distribution:",Counter(expanded_labels))

Expanded train size: 2000
Distribution: Counter({np.int64(0): 1014, np.int64(1): 986})


In [13]:
class PatchEmbedding(layers.Layer):

    def __init__(self, patch, dim):
        super().__init__()
        self.patch = patch
        self.dim = dim
        self.proj = layers.Conv2D(dim, patch, strides=patch)

    def build(self, input_shape):

        num_patches = (input_shape[1] // self.patch) ** 2

        self.cls = self.add_weight(
            shape=(1,1,self.dim),
            initializer="random_normal",
            trainable=True
        )

        self.pos = self.add_weight(
            shape=(1,num_patches+1,self.dim),
            initializer="random_normal",
            trainable=True
        )

    def call(self, x):

        x = self.proj(x)
        x = tf.reshape(x, [tf.shape(x)[0], -1, self.dim])

        batch = tf.shape(x)[0]
        cls = tf.repeat(self.cls, repeats=batch, axis=0)

        x = tf.concat([cls, x], axis=1)

        return x + self.pos

In [14]:
IMG_SIZE=224
BATCH_SIZE=8

def preprocess(path,label):

    img=cv2.imread(path.decode(),cv2.IMREAD_GRAYSCALE)
    img=cv2.resize(img,(IMG_SIZE,IMG_SIZE))
    img=img.astype(np.float32)/255.0
    img=np.expand_dims(img,-1)

    return img,np.float32(label)

def tf_preprocess(path,label):

    img,label=tf.py_function(
        preprocess,
        [path,label],
        [tf.float32,tf.float32]
    )

    img.set_shape((224,224,1))
    label.set_shape(())

    return img,label

def make_ds(paths,labels,train=False):

    ds=tf.data.Dataset.from_tensor_slices((paths,labels))

    if train:
        ds=ds.shuffle(2000)

    ds=ds.map(tf_preprocess,num_parallel_calls=tf.data.AUTOTUNE)
    ds=ds.batch(BATCH_SIZE)
    ds=ds.prefetch(tf.data.AUTOTUNE)

    return ds

train_ds=make_ds(expanded_paths,expanded_labels,True)
val_ds=make_ds(val_paths,y_val)
test_ds=make_ds(test_paths,y_test)

In [15]:
class TransformerBlock(layers.Layer):

    def __init__(self,dim):

        super().__init__()

        self.norm1=layers.LayerNormalization(epsilon=1e-6)

        self.attn=layers.MultiHeadAttention(
            num_heads=4,
            key_dim=dim//4
        )

        self.norm2=layers.LayerNormalization(epsilon=1e-6)

        self.mlp=keras.Sequential([
            layers.Dense(dim*4,activation="gelu"),
            layers.Dense(dim)
        ])

    def call(self,x):

        x=x+self.attn(self.norm1(x),self.norm1(x))
        x=x+self.mlp(self.norm2(x))

        return x

In [16]:
def build_model():

    inp=layers.Input((224,224,1))

    x=PatchEmbedding(4,526)(inp)

    x=TransformerBlock(526)(x)

    x=layers.Dense(256)(x)

    x=TransformerBlock(256)(x)

    x=layers.LayerNormalization()(x)

    cls=x[:,0]

    x=layers.Dense(32,activation="gelu")(cls)
    x=layers.Dense(16,activation="gelu")(x)
    x=layers.Dense(8,activation="gelu")(x)
    x=layers.Dense(4,activation="gelu")(x)
    x=layers.Dense(2,activation="gelu")(x)

    out=layers.Dense(1,activation="sigmoid")(x)

    return keras.Model(inp,out)

model=build_model()

In [17]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ patch_embedding                 │ (None, 3137, 526)      │     1,659,530 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 3137, 526)      │     3,322,736 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3137, 256)      │       134,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_1             │ (None, 3137, 256)      │       789,760 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_4           │ (None, 3137, 256)      │           512 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         8,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 4)              │            36 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 2)              │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,916,387 (22.57 MB)

 Trainable params: 5,916,387 (22.57 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(

    optimizer=tf.keras.optimizers.Adam(3e-5),

    loss="binary_crossentropy",

    metrics=[
        "accuracy",
        tf.keras.metrics.AUC(name="auc")
    ]
)

In [20]:
early=tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    patience=5,
    mode="max",
    restore_best_weights=True
)

history=model.fit(

    train_ds,

    validation_data=val_ds,

    epochs=40,

    callbacks=[early]
)

Epoch 1/40


UnknownError: Graph execution error:

Detected at node EagerPyFunc defined at (most recent call last):
<stack traces unavailable>
Detected at node EagerPyFunc defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) UNKNOWN:  AttributeError: 'tensorflow.python.framework.ops.EagerTensor' object has no attribute 'decode'
Traceback (most recent call last):

  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/ops/script_ops.py", line 267, in __call__
    return func(device, token, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/ops/script_ops.py", line 145, in __call__
    outputs = self._call(device, args)
              ^^^^^^^^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/ops/script_ops.py", line 152, in _call
    ret = self._func(*args)
          ^^^^^^^^^^^^^^^^^

  File "/usr/local/lib/python3.12/dist-packages/tensorflow/python/autograph/